<a href="https://colab.research.google.com/github/yaranoun/ML-Tech/blob/main/notebooks/03_rag_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [42]:
!pip install -q transformers accelerate sentence-transformers faiss-cpu

In [43]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

llm = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

In [44]:
!git clone https://github.com/yaranoun/ML-Tech.git

Cloning into 'ML-Tech'...
remote: Enumerating objects: 632, done.
remote: Counting objects: 100% (182/182), done.
remote: Compressing objects: 100% (169/169), done.
remote: Total 632 (delta 108), reused 26 (delta 13), pack-reused 450 (from 2)
Receiving objects: 100% (632/632), 589.04 KiB | 6.27 MiB/s, done.
Resolving deltas: 100% (383/383), done.


In [45]:
%cd /content/ML-Tech

/content/ML-Tech


In [46]:
!git pull origin main

remote: Enumerating objects: 72, done.
remote: Counting objects: 100% (72/72), done.
remote: Compressing objects: 100% (54/54), done.
remote: Total 57 (delta 38), reused 5 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (57/57), 19.65 KiB | 386.00 KiB/s, done.
From https://github.com/yaranoun/ML-Tech
 * branch            main       -> FETCH_HEAD
   42de673..7b38aee  main       -> origin/main
Updating 42de673..7b38aee
Fast-forward
 data/processed/chunks.json                         |   44 +-
 ...30\243\330\254\331\206\330\250\331\212\330\251" |    4 +-
 ...04\331\201\330\271\331\204\331\212\330\251.txt" |    4 +-
 ...54\330\252\331\205\330\247\330\271\331\212.txt" |    4 +-
 ...1\331\206 \330\266\330\247\330\246\330\271.txt" |    4 +-
 ...2 \330\257\331\210\331\204\331\212\330\251.txt" |    4 +-
 ...1 \330\247\331\204\330\263\331\210\331\202.txt" |    4 +-
 ...04\331\204\330\256\330\247\330\261\330\254.txt" |    4 +-
 notebooks/03_rag_pipeline.ipynb                    | 6037 +

In [47]:
import faiss
index = faiss.read_index("data/processed/passport_index.faiss")

In [48]:
import json

with open("data/processed/chunks.json", "r", encoding="utf-8") as f:
    chunks = json.load(f)

In [49]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "intfloat/multilingual-e5-base"
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [50]:
def retrieve(question, k=3):
    query_embedding = embedding_model.encode(
        ["query: " + question],
        normalize_embeddings=True
    )

    scores, indices = index.search(query_embedding, k)

    results = []

    for score, idx in zip(scores[0], indices[0]):
        results.append({
            "score": float(score),
            "document": chunks[idx]["document"],
            "service":chunks[idx].get("service",""),
            "section": chunks[idx]["section"],
            "language":chunks[idx].get("language",""),
            "text": chunks[idx]["text"],
            "url": chunks[idx]["url"]
        })

    return results

In [51]:
results = retrieve(
    "ما هي المستندات لتجديد دفتر القيادة الخصوصي؟"
)

for r in results:
    print(r["score"], r["service"], r["section"])

0.8270811438560486 تجديد رخصة سوق منتهية الصلاحة المستندات المطلوبة
0.8176745772361755 تجديد رخصة سوق منتهية الصلاحة الوصف
0.8095848560333252 تجديد رخصة قيادة ضائعة المستندات المطلوبة


In [52]:
results = retrieve(
    "ما هي المستندات لتجديد دفتر القيادة الخصوصي؟",
    k=3
)
for result in results:
    print(result["section"], result["score"])

المستندات المطلوبة 0.8270811438560486
الوصف 0.8176745772361755
المستندات المطلوبة 0.8095848560333252


In [53]:
!pip install -q transformers accelerate

In [54]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

llm_name = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(llm_name)

llm = AutoModelForCausalLM.from_pretrained(
    llm_name,
    torch_dtype="auto",
    device_map="auto"
)

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

In [55]:
question = "ما هي المستندات لتجديد دفتر القيادة الخصوصي؟"

results = retrieve(question, k=3)

In [56]:
context = "\n\n".join(
    f"Section: {result['section']}\n{result['text']}"
    for result in results
)

print(context)

Section: المستندات المطلوبة
2.1
رخصة السوق المنتهية الصالحية

2.2
سجل عدلي لا يعود تاريخه لأكثر من ثلاثة اشهر

2.3
بطاقة أو إفادة فئة الدم

2.4
شهادة طبية صادرة من نقابة الاطباء، وعليها الرسم الشمسي لصاحب العلاقة (لا يعود تاريخها لأكثر من ثلاثة اشهر)

2.5
صورة عن بطاقة الهوية أو إخراح قيد أو صورة عن جواز السفر. يجب على غير اللبنانيين ابراز اقامة صالحة.

2.6
صورة شمسية مصدقة من المختار (عدد2)

3

Section: الوصف
(يستوجب حضور صاحب العلاقة شخصيا)
لا يتطلب موعد
 عند انتهاء تاريخ صلاحية رخصة السوق(خصوصية أوعمومية) يتوجب على المواطن تجديدها في المركز الذي صدرت عنه.

2

Section: المستندات المطلوبة
:

2.1
رخص السوق المفقودة الغير منتهية الصلاحية:

2.1.1
صورة طبق الاصل للمحضر المنظم لدى قوى الامن الداخلي.

2.1.2
صورة عن بطاقة الهوية أو إخراح قيد أو صورة عن جواز السفر. يجب على غير اللبنانيين ابراز اقامة صالحة.

2.1.3
صورة شمسية مصدقة من المختار (عدد2)

2.2
رخصة السوق المفقودة منتهية الصلاحية (بالإضافة إلى المستندات أعلاه):

2.2.1
شهادة طبية صادرة من نقابة الاطباء، وعليها الرسم الشمسي لصاحب العلاق

In [57]:
def ask(question, k=3):

    # 1. Retrieve relevant chunks
    results = retrieve(question, k=k)

    # 2. Build context for the LLM
    context = "\n\n".join(
    f"""
Service: {result.get('service', '')}
Section: {result['section']}
Language: {result.get('language', '')}
Content:
{result['text']}
"""
    for result in results
)

    # 3. Create the prompt
    messages = [
       {
    "role": "system",
    "content": """
You are an assistant for Lebanese government procedures.

Answer the user's question using ONLY the information contained in the
provided official government context.

IMPORTANT:
- The context may be in Arabic, English, or French.
- The user's question may be in Arabic, English, or French.
- Answer in the same language as the user's question.
- Carefully read information written in Arabic.
- Use relevant information even if the wording of the question is different
  from the wording in the context.
- Do not require an exact phrase match between the question and the context.
- The retrieved context may contain irrelevant passages. Ignore those passages.
- If relevant information is present in any retrieved passage, use it to answer.
- Include all relevant documents, requirements, fees, and conditions found
  in the context.
- Do not invent information that is not present in the context.
- Only say that the information could not be found if NONE of the retrieved
  context contains information that answers the question.

Answer clearly and concisely.
"""
        },
        {
            "role": "user",
            "content": f"""
Official context:

{context}

Question:
{question}
"""
        }
    ]

    # 4. Convert messages into Qwen's chat format
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(llm.device)

    # 5. Generate answer
    with torch.no_grad():
        outputs = llm.generate(
            **inputs,
            max_new_tokens=300,
            do_sample=False
        )

    generated_tokens = outputs[0][inputs.input_ids.shape[1]:]

    answer = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    # 6. Collect unique sources
    sources = []

    for result in results:
        url = result["url"]

        if url and url not in sources:
            sources.append(url)

    return {
        "answer": answer,
        "sources": sources,
        "retrieved_chunks": results
    }

In [58]:
test_questions = [

    # -------------------------
    # ENGLISH - PASSPORT
    # -------------------------

    {
        "id": 1,
        "language": "en",
        "question": "What documents do I need to apply for a biometric passport?",
        "expected_service": "Biometric Passport",
        "expected_section": "Requested documents"
    },

    {
        "id": 2,
        "language": "en",
        "question": "How much does a 10-year passport cost?",
        "expected_service": "Biometric Passport",
        "expected_section": "Fees"
    },

    {
        "id": 3,
        "language": "en",
        "question": "What are the fees for a 5-year passport?",
        "expected_service": "Biometric Passport",
        "expected_section": "Fees"
    },

    {
        "id": 4,
        "language": "en",
        "question": "What should I do if I lose my passport?",
        "expected_service": "Lost or Stolen Passport",
        "expected_section": "Lost Passport"
    },

    {
        "id": 5,
        "language": "en",
        "question": "My passport was stolen. What do I need to do?",
        "expected_service": "Lost or Stolen Passport",
        "expected_section": "Stolen Passport"
    },

    {
        "id": 6,
        "language": "en",
        "question": "Do I have to report a stolen passport if I don't want a new one?",
        "expected_service": "Lost or Stolen Passport",
        "expected_section": "NB"
    },

    {
        "id": 7,
        "language": "en",
        "question": "Do I need to go in person to apply for a passport?",
        "expected_service": "Personal attendance required for Passport",
        "expected_section": "Personal attendance required"
    },

    {
        "id": 8,
        "language": "en",
        "question": "Can a sick person who cannot leave home apply for a passport?",
        "expected_service": "Personal attendance required for Passport",
        "expected_section": "Exemption from attendance"
    },

    {
        "id": 9,
        "language": "en",
        "question": "How can I get a copy of my passport certified?",
        "expected_service": "Certifying Passport",
        "expected_section": "Full Document"
    },

    {
        "id": 10,
        "language": "en",
        "question": "What documents are required to export a Lebanese passport?",
        "expected_service": "Exporting biometric passport",
        "expected_section": "Exporting a Lebanese passport"
    },


    # -------------------------
    # ARABIC - DRIVING LICENCE
    # -------------------------

    {
        "id": 11,
        "language": "ar",
        "question": "ما هي المستندات المطلوبة لتجديد رخصة سوق منتهية الصلاحية؟",
        "expected_service": "تجديد رخصة سوق منتهية الصلاحة",
        "expected_section": "المستندات المطلوبة"
    },

    {
        "id": 12,
        "language": "ar",
        "question": "ما هي الأوراق التي أحتاجها لتجديد دفتر القيادة الخصوصي؟",
        "expected_service": "تجديد رخصة سوق منتهية الصلاحة",
        "expected_section": "المستندات المطلوبة"
    },

    {
        "id": 13,
        "language": "ar",
        "question": "انتهت صلاحية رخصة السوق، كيف أجددها؟",
        "expected_service": "تجديد رخصة قيادة ضائعة",
        "expected_section": "المستندات المطلوبة"
    },

    {
        "id": 14,
        "language": "ar",
        "question": "ما هي المستندات المطلوبة في حال ضياع رخصة القيادة؟",
        "expected_service": "تجديد رخصة قيادة ضائعة",
        "expected_section": "المستندات المطلوبة"
    },

    {
        "id": 15,
        "language": "ar",
        "question": "ضيعت دفتر السوق، ماذا يجب أن أفعل؟",
        "expected_service": "تجديد رخصة قيادة ضائعة",
        "expected_section": "المستندات المطلوبة"
    },

    {
        "id": 16,
        "language": "ar",
        "question": "كيف يمكنني استبدال رخصة سوق أجنبية؟",
        "expected_service": "استبدال رخص السوق الأجنبية",
        "expected_section": "الوصف"
    },

    {
        "id": 17,
        "language": "ar",
        "question": "ما هي الأوراق اللازمة لاستبدال رخصة قيادة أجنبية؟",
        "expected_service": "استبدال رخص السوق الأجنبية",
        "expected_section": "المستندات المطلوبة"
    }
]

In [59]:
for test in test_questions:

    results = retrieve(test["question"], k=3)

    top = results[0]

    service_correct = (
        top["service"].strip() ==
        test["expected_service"].strip()
    )

    section_correct = (
        top["section"].strip() ==
        test["expected_section"].strip()
    )

    correct = service_correct and section_correct

    print("=" * 60)
    print("Question:", test["question"])
    print("Expected:", test["expected_service"], "|", test["expected_section"])
    print("Retrieved:", top["service"], "|", top["section"])
    print("Score:", top.get("rerank_score", top.get("score")))
    print("Correct:", correct)

Question: What documents do I need to apply for a biometric passport?
Expected: Biometric Passport | Requested documents
Retrieved: Biometric Passport | Requested documents
Score: 0.8623675107955933
Correct: True
Question: How much does a 10-year passport cost?
Expected: Biometric Passport | Fees
Retrieved: Biometric Passport | Fees
Score: 0.8490024209022522
Correct: True
Question: What are the fees for a 5-year passport?
Expected: Biometric Passport | Fees
Retrieved: Biometric Passport | Fees
Score: 0.8650145530700684
Correct: True
Question: What should I do if I lose my passport?
Expected: Lost or Stolen Passport | Lost Passport
Retrieved: Lost or Stolen Passport | Lost Passport
Score: 0.8587358593940735
Correct: True
Question: My passport was stolen. What do I need to do?
Expected: Lost or Stolen Passport | Stolen Passport
Retrieved: Lost or Stolen Passport | Stolen Passport
Score: 0.8491607904434204
Correct: True
Question: Do I have to report a stolen passport if I don't want a new

In [60]:
response = ask("ما هي المستندات لتجديد دفتر القيادة الخصوصي؟")

In [61]:
print(response["answer"])

print("\nSources:")
for source in response["sources"]:
    print(source)

لتجديد دفتر القيادة الخصوصي، المستندات المطلوبة هي:

2.1
رخصة السوق المفقودة الغير منتهية الصلاحية:
   - صورة طبق الأصل للمحضر المنظم لدى قوى الأمن الداخلي.
   - صورة عن بطاقة الهوية أو إخراج قيد أو صورة عن جواز السفر. يجب على غير اللبنانيين إبراز إقامة صالحة.
   - صورة شمسية مصدقة من المختار (عدد 2).

2.2
رخصة السوق المفقودة منتهية الصلاحية (بالإضافة إلى المستندات أعلاه):
   - شهادة طبية صادرة من نقابة الأطباء، وعليها الرسم الشمسي لصاحب العلاقة (لا يعود تاريخها لأكثر من ثلاثة أشهر).
   - سجل عدلي لا يعود تاريخه لأكثر من ثلاثة أشهر.
   - بطاقة أو إفادة فئة الدم.

Sources:
https://tmo.gov.lb/web/panel/info/service-types/1
